# ECP → YOLOv8 Fine-tuning + OSNet-AIN Embedding Extraction
**Thesis: Multi-Camera Re-Identification System**

This notebook:
1. Mounts Google Drive (all outputs persist there)
2. Downloads the EuroCity Persons dataset
3. Converts ECP annotations → YOLO format
4. Fine-tunes `yolov8n.pt` on ECP pedestrians
5. Extracts OSNet-AIN embeddings from ECP crops

**Runtime:** GPU (T4 or better) — Runtime → Change runtime type → T4 GPU

## 0. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

# All work lives under this folder in your Drive
WORK_DIR = '/content/drive/MyDrive/Thesis/ReID'
ECP_DIR  = f'{WORK_DIR}/ECP'
YOLO_DATASET_DIR = f'{WORK_DIR}/datasets/ecp_yolo'
CROPS_DIR = f'{WORK_DIR}/ecp_crops'

os.makedirs(ECP_DIR, exist_ok=True)
os.makedirs(YOLO_DATASET_DIR, exist_ok=True)
os.makedirs(CROPS_DIR, exist_ok=True)

print('Drive mounted. Working directory:', WORK_DIR)

## 1. Install Dependencies

In [ ]:
# Install all required packages
!pip install -q ultralytics
!pip install -q torch torchvision
!pip install -q opencv-python-headless

# torchreid — install from source (not on PyPI)
!pip install -q Cython
!git clone -q https://github.com/KaiyangZhou/deep-person-reid.git /content/deep-person-reid
%cd /content/deep-person-reid
!pip install -q -r requirements.txt
!python setup.py develop -q
%cd /content

print('All dependencies installed.')

## 2. Download ECP Dataset

Enter your credentials from **eurocity-dataset.tudelft.nl**.  
Files are saved to Drive so you only need to download once.

In [ ]:
from getpass import getpass

ECP_USER = input('ECP username: ')
ECP_PASS = getpass('ECP password: ')

In [ ]:
import os

BASE_URL = 'http://eurocity-dataset.tudelft.nl//eval/downloadFiles/downloadFile/detection?file=ecpdata%2Fecpdataset_v1%2F'

# Files to download — comment out night/* if you only want day
ECP_FILES = [
    'ECP_day_img_train.zip',
    'ECP_day_img_val.zip',
    'ECP_day_labels_train.zip',
    'ECP_day_labels_val.zip',
    # 'ECP_night_img_train.zip',
    # 'ECP_night_img_val.zip',
    # 'ECP_night_labels_train.zip',
    # 'ECP_night_labels_val.zip',
]

for fname in ECP_FILES:
    dest = f'{ECP_DIR}/{fname}'
    if os.path.exists(dest):
        print(f'[skip] {fname} already downloaded')
        continue
    url = BASE_URL + fname
    print(f'Downloading {fname} ...')
    !wget --auth-no-challenge \
          --user="{ECP_USER}" \
          --password="{ECP_PASS}" \
          --output-document="{dest}" \
          "{url}"
    print(f'  Saved to {dest}')

In [ ]:
# Extract zips (skip if already extracted)
import zipfile, os

for fname in ECP_FILES:
    zip_path = f'{ECP_DIR}/{fname}'
    if not os.path.exists(zip_path):
        print(f'[skip] {fname} not found')
        continue
    # Check if already extracted by looking for expected subfolder
    split = 'day' if 'day' in fname else 'night'
    kind  = 'img' if 'img' in fname else 'labels'
    subset = 'train' if 'train' in fname else 'val'
    expected = f'{ECP_DIR}/ECP/{split}/{kind}/{subset}'
    if os.path.exists(expected):
        print(f'[skip] {fname} already extracted')
        continue
    print(f'Extracting {fname} ...')
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(ECP_DIR)
    print(f'  Done.')

ECP_ROOT = f'{ECP_DIR}/ECP'
print('ECP root:', ECP_ROOT)
!ls "{ECP_ROOT}"

## 3. Upload Project Scripts

Upload `ecp_to_yolo.py` and `ecp_extract_crops.py` from your laptop, **or** clone your repo if it's on GitHub.

In [ ]:
# Option A: Upload from laptop
from google.colab import files
print('Select ecp_to_yolo.py and ecp_extract_crops.py from your laptop:')
uploaded = files.upload()
for name in uploaded:
    print(f'  Uploaded: {name}')

In [ ]:
# Option B: Clone from GitHub (uncomment and replace URL if your repo is on GitHub)
# !git clone https://github.com/YOUR_USERNAME/YOUR_REPO.git /content/reid_repo
# !cp /content/reid_repo/ecp_to_yolo.py /content/
# !cp /content/reid_repo/ecp_extract_crops.py /content/

## 4. Convert ECP → YOLO Format

In [ ]:
!python ecp_to_yolo.py \
    --ecp-root "{ECP_ROOT}" \
    --output-dir "{YOLO_DATASET_DIR}" \
    --split day \
    --min-height 20 \
    --copy-images

# Verify output
!ls "{YOLO_DATASET_DIR}"
!cat "{YOLO_DATASET_DIR}/ecp.yaml"

In [ ]:
# Quick sanity check: count images and labels
import glob
train_imgs   = glob.glob(f'{YOLO_DATASET_DIR}/images/train/*.png')
train_labels = glob.glob(f'{YOLO_DATASET_DIR}/labels/train/*.txt')
val_imgs     = glob.glob(f'{YOLO_DATASET_DIR}/images/val/*.png')
print(f'Train: {len(train_imgs):,} images, {len(train_labels):,} label files')
print(f'Val:   {len(val_imgs):,} images')

## 5. Fine-tune YOLOv8n on ECP

Training saves best weights to Drive automatically via the `project` argument.

In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8n.pt')  # start from COCO pretrained

results = model.train(
    data    = f'{YOLO_DATASET_DIR}/ecp.yaml',
    epochs  = 50,
    imgsz   = 640,
    batch   = 16,          # reduce to 8 if OOM
    device  = 0,           # GPU
    project = f'{WORK_DIR}/runs/detect',
    name    = 'ecp_yolov8n',
    exist_ok= True,
    patience= 10,          # early stopping
    workers = 2,
)

In [ ]:
# Evaluate fine-tuned model on ECP val set
best_weights = f'{WORK_DIR}/runs/detect/ecp_yolov8n/weights/best.pt'
fine_tuned   = YOLO(best_weights)
metrics      = fine_tuned.val(data=f'{YOLO_DATASET_DIR}/ecp.yaml', device=0)

print('\n=== Fine-tuned YOLOv8n on ECP ===')
print(f'  mAP50:    {metrics.box.map50:.4f}')
print(f'  mAP50-95: {metrics.box.map:.4f}')
print(f'  Precision:{metrics.box.p[0]:.4f}')
print(f'  Recall:   {metrics.box.r[0]:.4f}')

In [ ]:
# Compare baseline (COCO pretrained) vs fine-tuned on ECP val
baseline = YOLO('yolov8n.pt')
base_metrics = baseline.val(data=f'{YOLO_DATASET_DIR}/ecp.yaml', device=0)

print('\n=== Baseline YOLOv8n (COCO pretrained) ===')
print(f'  mAP50:    {base_metrics.box.map50:.4f}')
print(f'  mAP50-95: {base_metrics.box.map:.4f}')
print(f'  Precision:{base_metrics.box.p[0]:.4f}')
print(f'  Recall:   {base_metrics.box.r[0]:.4f}')

print('\n=== Delta (fine-tuned - baseline) ===')
print(f'  ΔmAP50:    {metrics.box.map50 - base_metrics.box.map50:+.4f}')
print(f'  ΔmAP50-95: {metrics.box.map   - base_metrics.box.map:+.4f}')

## 6. Extract Person Crops + OSNet-AIN Embeddings

In [ ]:
!python ecp_extract_crops.py \
    --ecp-root "{ECP_ROOT}" \
    --output-dir "{CROPS_DIR}" \
    --split day \
    --subset val \
    --min-height 64 \
    --embed \
    --embed-batch 64

In [ ]:
# Visualise embedding space with t-SNE
import numpy as np
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE

data = np.load(f'{CROPS_DIR}/embeddings.npz', allow_pickle=True)
embeddings = data['embeddings']   # [N, 512]
paths      = data['paths']

print(f'Embeddings shape: {embeddings.shape}')

# Subsample for speed if large
MAX_POINTS = 5000
if len(embeddings) > MAX_POINTS:
    idx = np.random.choice(len(embeddings), MAX_POINTS, replace=False)
    embeddings_sub = embeddings[idx]
    paths_sub = paths[idx]
else:
    embeddings_sub = embeddings
    paths_sub = paths

# Color by city (extracted from filename)
cities = [os.path.basename(p).split('_')[1] for p in paths_sub]
unique_cities = sorted(set(cities))
city_to_idx = {c: i for i, c in enumerate(unique_cities)}
colors = [city_to_idx[c] for c in cities]

print(f'Running t-SNE on {len(embeddings_sub)} embeddings ...')
z = TSNE(n_components=2, perplexity=30, random_state=42).fit_transform(embeddings_sub)

fig, ax = plt.subplots(figsize=(12, 8))
sc = ax.scatter(z[:, 0], z[:, 1], c=colors, cmap='tab20', s=4, alpha=0.7)
plt.colorbar(sc, ax=ax, label='City')
ax.set_title('OSNet-AIN Embedding Space on ECP Val (coloured by city)')
ax.set_xlabel('t-SNE dim 1')
ax.set_ylabel('t-SNE dim 2')
plt.tight_layout()
plt.savefig(f'{WORK_DIR}/tsne_ecp_embeddings.png', dpi=150)
plt.show()
print(f'Saved: {WORK_DIR}/tsne_ecp_embeddings.png')

## 7. Download Fine-tuned Weights to Laptop

Download `best.pt` to your local machine, then update `edge_client.py`:
```python
detector = YOLO('ecp_yolov8n_best.pt')   # instead of 'yolov8n.pt'
```

In [ ]:
from google.colab import files

best_pt = f'{WORK_DIR}/runs/detect/ecp_yolov8n/weights/best.pt'
print(f'Downloading {best_pt} ...')
files.download(best_pt)

## 8. (Optional) Run evaluate.py in Colab

If you have a `ground_truth.csv` annotated from ECP crops, upload it and run evaluation here.

In [ ]:
# Upload ground_truth.csv and evaluate.py + edge_client.py from your laptop
from google.colab import files
print('Upload ground_truth.csv, evaluate.py, and edge_client.py:')
files.upload()

In [ ]:
!python evaluate.py \
    --gt ground_truth.csv \
    --category person \
    --output "{WORK_DIR}/eval_results.csv"

# Download results
files.download(f'{WORK_DIR}/eval_results.csv')